# Qwen3-ASR Speech Recognition with OpenVINO™

The Qwen3-ASR family includes Qwen3-ASR-1.7B and Qwen3-ASR-0.6B, which support language identification and ASR for 52 languages and dialects. Both leverage large-scale speech training data and the strong audio understanding capability of their foundation model, Qwen3-Omni. Experiments show that the 1.7B version achieves state-of-the-art performance among open-source ASR models and is competitive with the strongest proprietary commercial APIs. Here are the main features:

* **All-in-one**: Qwen3-ASR-1.7B and Qwen3-ASR-0.6B support language identification and speech recognition for 30 languages and 22 Chinese dialects, so as to English accents from multiple countries and regions.

* **Excellent and Fast**: The Qwen3-ASR family ASR models maintains high-quality and robust recognition under complex acoustic environments and challenging text patterns. Qwen3-ASR-1.7B achieves strong performance on both open-sourced and internal benchmarks. While the 0.6B version achieves accuracy-efficient trade-off, it reaches 2000 times throughput at a concurrency of 128. They both achieve streaming / offline unified inference with single model and support transcribe long audio.

* **Novel and strong forced alignment Solution**: We introduce Qwen3-ForcedAligner-0.6B, which supports timestamp prediction for arbitrary units within up to 5 minutes of speech in 11 languages. Evaluations show its timestamp accuracy surpasses E2E based forced-alignment models.

* **Comprehensive inference toolkit**: In addition to open-sourcing the architectures and weights of the Qwen3-ASR series, we also release a powerful, full-featured inference framework that supports vLLM-based batch inference, asynchronous serving, streaming inference, timestamp prediction, and more.

<p align="center">
    <img src="https://qianwen-res.oss-cn-beijing.aliyuncs.com/Qwen3-ASR-Repo/overview.jpg" width="100%"/>
<p>

More details can be found in the original [repository](https://github.com/QwenLM/Qwen3-ASR) and [model card](https://huggingface.co/Qwen/Qwen3-ASR-0.6B)

In this tutorial, we will:
1. Install required dependencies
2. Download and convert Qwen3-ASR model to OpenVINO format
3. Create an inference pipeline using OpenVINO
4. Build an interactive Gradio demo for speech recognition

#### Table of contents:

- [Prerequisites](#Prerequisites)
- [Download and Prepare Qwen3-ASR Model](#Download-and-Prepare-Qwen3-ASR-Model)
- [Convert Model to OpenVINO Format](#Convert-Model-to-OpenVINO-Format)
- [Create Inference Pipeline](#Create-Inference-Pipeline)
    - [Select Inference Device](#Select-Inference-Device)
    - [Run Speech Recognition](#Run-Speech-Recognition)
- [Interactive Demo](#Interactive-Demo)


⚠️ **EXPERIMENTAL NOTEBOOK**

This notebook demonstrates a model that has not been fully validated with OpenVINO. It may be fully supported and validated in the future.

<img referrerpolicy="no-referrer-when-downgrade" src="https://static.scarf.sh/a.png?x-pxid=5b5a4db0-7875-4bfb-bdbd-01698b5b1a77&file=notebooks/qwen3-asr/qwen3-asr.ipynb" />

### Installation Instructions

This is a self-contained example that relies solely on its own code.

We recommend  running the notebook in a virtual environment. You only need a Jupyter server to start.
For details, please refer to [Installation Guide](https://github.com/openvinotoolkit/openvino_notebooks/blob/latest/README.md#-installation-guide).

## Prerequisites
[back to top ⬆️](#Table-of-contents:)

In [ ]:
# Fetch notebook_utils module
import requests
from pathlib import Path

if not Path("notebook_utils.py").exists():
    r = requests.get(
        url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/notebook_utils.py",
    )
    open("notebook_utils.py", "w").write(r.text)

if not Path("cmd_helper.py").exists():
    r = requests.get(
        url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/cmd_helper.py",
    )
    open("cmd_helper.py", "w").write(r.text)

if not Path("pip_helper.py").exists():
    r = requests.get(
        url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/pip_helper.py",
    )
    open("pip_helper.py", "w").write(r.text)

# Read more about telemetry collection at https://github.com/openvinotoolkit/openvino_notebooks?tab=readme-ov-file#-telemetry
from notebook_utils import collect_telemetry

collect_telemetry("qwen3-asr.ipynb")

### Install Dependencies
[back to top ⬆️](#Table-of-contents:)

Install required packages for Qwen3-ASR model conversion and inference:

In [ ]:
from cmd_helper import clone_repo
from pip_helper import pip_install

# Uninstall conflicting versions
%pip uninstall -y -q torch torchvision torchaudio transformers qwen-asr

pip_install(
    "-q",
    "--extra-index-url",
    "https://download.pytorch.org/whl/cpu",
    "torch==2.8.0",
    "nncf",
    "torchaudio==2.8.0",
    "openvino>=2025.4.0",
    "gradio>=4.0",
    "huggingface_hub",
    "scipy",
    "qwen-asr",
)

# Clone Qwen3-ASR repository for model code
repo_dir = Path("Qwen3-ASR")
revision = "c17a131fe028b2e428b6e80a33d30bb4fa57b8df"
clone_repo("https://github.com/QwenLM/Qwen3-ASR.git", revision)

pip_install(
    "-q -e",
    str(repo_dir),
)

## Download and Prepare Qwen3-ASR Model
[back to top ⬆️](#Table-of-contents:)

Select the Qwen3-ASR model variant to use. The 0.6B model is recommended for faster inference while maintaining good accuracy:

In [3]:
import ipywidgets as widgets

model_ids = [
    "Qwen/Qwen3-ASR-0.6B",
    "Qwen/Qwen3-ASR-1.7B",
]

model_selector = widgets.Dropdown(
    options=model_ids,
    value=model_ids[0],
    description="Model:",
)

model_selector

Dropdown(description='Model:', options=('Qwen/Qwen3-ASR-0.6B', 'Qwen/Qwen3-ASR-1.7B'), value='Qwen/Qwen3-ASR-0…

### Model Architecture

The Qwen3-ASR pipeline consists of several key components:

* **Audio Frontend**: Extracts mel-spectrogram features from raw audio waveform (16kHz sample rate, 80 mel bins)
* **Audio Encoder (Audio Tower)**: Conv2D layers followed by Transformer encoder blocks that process audio features into embeddings
* **Text Embeddings**: Converts text tokens into embeddings for the language model
* **Language Model (LLM)**: A Qwen3-based decoder that takes merged audio and text embeddings and generates transcription output

The model uses a multimodal approach where audio embeddings replace special audio tokens in the text sequence before being processed by the LLM.

## Convert Model to OpenVINO Format
[back to top ⬆️](#Table-of-contents:)

Now we'll convert the Qwen3-ASR model to OpenVINO Intermediate Representation (IR) format. The conversion process exports the following components:

1. **Audio Conv Model** (`openvino_thinker_audio_model.xml`): The Conv2D frontend for audio feature extraction
2. **Audio Encoder Model** (`openvino_thinker_audio_encoder_model.xml`): The Transformer encoder layers
3. **Embedding Model** (`openvino_thinker_embedding_model.xml`): Text token embedding layer
4. **Language Model** (`openvino_thinker_language_model.xml`): The main LLM decoder with KV-cache support

In [ ]:
from qwen_3_asr_helper import convert_qwen3_asr_model

# from nncf import CompressWeightsMode

model_id = model_selector.value
model_name = model_id.split("/")[-1]
ov_model_dir = Path(f"{model_name}-OV")

# Convert model to OpenVINO format
# This will skip conversion if the model already exists
convert_qwen3_asr_model(
    model_id=model_id,
    output_dir=ov_model_dir,
    quantization_config=None,  # Set to {"mode": CompressWeightsMode.INT8_SYM} for INT8 quantization
)

✅ Qwen/Qwen3-ASR-0.6B model already converted. You can find results in Qwen3-ASR-0.6B-OV


## Create Inference Pipeline
[back to top ⬆️](#Table-of-contents:)

### Select Inference Device
[back to top ⬆️](#Table-of-contents:)

Select the device for running inference. CPU provides the most compatibility, while GPU and NPU can provide acceleration on supported hardware:

In [ ]:
from notebook_utils import device_widget

device = device_widget("CPU", exclude=["NPU"])

device

Dropdown(description='Device:', options=('CPU', 'GPU'), value='CPU')

### Load OpenVINO Model
[back to top ⬆️](#Table-of-contents:)

Load the converted OpenVINO model using our helper class. The `OVQwen3ASRModel` provides the same API as the original Qwen3ASRModel:

In [6]:
from qwen_3_asr_helper import OVQwen3ASRModel

ov_model = OVQwen3ASRModel.from_pretrained(
    model_dir=str(ov_model_dir),
    device=device.value,
    max_inference_batch_size=32,  # Batch size limit for inference. -1 means unlimited.
    max_new_tokens=256,  # Maximum number of tokens to generate. Set larger for long audio.
)

# Print supported languages
print(f"Supported languages: {ov_model.get_supported_languages()}")

⌛ Loading audio conv model from Qwen3-ASR-0.6B-OV/thinker/openvino_thinker_audio_model.xml
⌛ Loading audio encoder model from Qwen3-ASR-0.6B-OV/thinker/openvino_thinker_audio_encoder_model.xml
⌛ Loading embedding model from Qwen3-ASR-0.6B-OV/thinker/openvino_thinker_embedding_model.xml
⌛ Loading language model from Qwen3-ASR-0.6B-OV/thinker/openvino_thinker_language_model.xml


The tokenizer you are loading from 'Qwen3-ASR-0.6B-OV' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


✅ Processor loaded successfully
✅ OVQwen3ASRModel initialized successfully
Supported languages: ['Chinese', 'English', 'Cantonese', 'Arabic', 'German', 'French', 'Spanish', 'Portuguese', 'Indonesian', 'Italian', 'Korean', 'Russian', 'Thai', 'Vietnamese', 'Japanese', 'Turkish', 'Hindi', 'Malay', 'Dutch', 'Swedish', 'Danish', 'Finnish', 'Polish', 'Czech', 'Filipino', 'Persian', 'Greek', 'Romanian', 'Hungarian', 'Macedonian']


### Run Speech Recognition
[back to top ⬆️](#Table-of-contents:)

Let's test the model with sample audio files. The model supports various audio formats (wav, mp3, flac) and can accept:
- Local file paths
- URLs to audio files
- NumPy arrays with sample rate tuples

In [7]:
# Download sample audio files
import urllib.request

sample_audio_en = Path("sample_en.wav")

if not sample_audio_en.exists():
    print("Downloading English sample audio...")
    urllib.request.urlretrieve("https://qianwen-res.oss-cn-beijing.aliyuncs.com/Qwen3-ASR-Repo/asr_en.wav", sample_audio_en)
    print(f"Downloaded: {sample_audio_en}")

results = ov_model.transcribe(
    audio=str(sample_audio_en),
    language=None,  # Auto-detect language
)

print(f"Detected Language: {results[0].language}")
print(f"Transcription: {results[0].text}")

Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


Detected Language: English
Transcription: Oh yeah, yeah, he wasn't that bad when I started listening to him. But his solo music didn't do overly well, but he did very well when started writing for other people.


## Interactive Demo
[back to top ⬆️](#Table-of-contents:)

Launch an interactive Gradio demo that allows you to:
- Upload audio files (WAV, MP3, FLAC, etc.)
- Record audio from your microphone
- Select target language or use auto-detection
- View transcription results with inference metrics

The demo is based on the official [Qwen3-ASR Hugging Face Space](https://huggingface.co/spaces/Qwen/Qwen3-ASR).

In [ ]:
from gradio_helper import make_demo

# Create Gradio demo
demo = make_demo(ov_model, example_dir=".")

# Launch the demo
# If you are launching remotely, specify server_name and server_port:
#   demo.launch(server_name='your_server_name', server_port=7860)
# If you have any issues launching, try:
#   demo.launch(share=True)

try:
    demo.launch(debug=True)
except Exception:
    demo.launch(debug=True, share=True)